In [1]:
import pytorch_lightning as pl
from pytorch_lightning.loggers import TensorBoardLogger
from pytorch_lightning import Trainer
from policy import ForwardPolicy, BackwardPolicy
from gflownet.gflownet import GFlowNet
from gflownet.dataset import MatrixDataModule
import itertools
from pytorch_lightning.callbacks import EarlyStopping, ModelCheckpoint
from gflownet.gflownet import GFlowNet
from gflownet.dataset import MatrixDataModule
import time
import os


def run_experiment(hyperparams):
    num_workers = os.cpu_count()
    start_time = time.time()
    matrix_dir = 'data/small_ILU'
    data_module = MatrixDataModule(matrix_directory=matrix_dir, num_workers=num_workers, batch_size=1)

    forward_policy = ForwardPolicy(node_features=hyperparams['node_features'], hidden_dim=hyperparams['hidden_dim'], max_num_actions=hyperparams['max_num_actions'])
    backward_policy = BackwardPolicy(input_dim=hyperparams['input_dim'], hidden_dim=hyperparams['hidden_dim'], max_num_actions=hyperparams['max_num_actions'])

    model = GFlowNet(forward_policy=forward_policy, backward_policy=backward_policy, no_sampling_batch=hyperparams['no_sampling_batch'], lr=hyperparams['lr'], schedule_patience=hyperparams['schedule_patience'])

    logger = TensorBoardLogger("tb_logs", name=f"gflownet_lr_{hyperparams['lr']}_epochs_{hyperparams['number_epoch']}_sampling_{hyperparams['no_sampling_batch']}_patience_{hyperparams['schedule_patience']}")

    callbacks = [
        EarlyStopping(monitor="train_loss", mode="min", patience=10),
        ModelCheckpoint(monitor="train_loss", save_top_k=3, mode="min")
    ]

    trainer = pl.Trainer(max_epochs=hyperparams['number_epoch'], logger=logger, callbacks=callbacks)
    trainer.fit(model, data_module)
    training_time = time.time() - start_time
    print(f"Elapsed Training Time: {training_time}")

if __name__ == '__main__':
    # Define hyperparameters space
    learning_rates = [2e-5]
    number_epochs = [50] #Change to 50, 100 after testing
    no_sampling_batches = [1] #Change to 4, 8, 16 after testing
    schedule_patience = [5] 

    # Create hyperparameter combinations
    hyperparams_combinations = list(itertools.product(learning_rates, number_epochs, no_sampling_batches, schedule_patience))

    # Run experiments for each combination
    for lr, number_epoch, no_sampling_batch, patience in hyperparams_combinations:
        hyperparams = {
            'lr': lr,
            'number_epoch': number_epoch,
            'no_sampling_batch': no_sampling_batch,
            'hidden_dim': 2,
            'node_features': -1,
            'input_dim': 1,
            'max_num_actions': 180000,
            'schedule_patience': patience
        }
        run_experiment(hyperparams)

GPU available: False, used: False
TPU available: False, using: 0 TPU cores
HPU available: False, using: 0 HPUs
/Users/tonylizza/opt/anaconda3/envs/ML_new/lib/python3.12/site-packages/pytorch_lightning/trainer/configuration_validator.py:68: You passed in a `val_dataloader` but have no `validation_step`. Skipping val loop.
/Users/tonylizza/opt/anaconda3/envs/ML_new/lib/python3.12/site-packages/torch/optim/lr_scheduler.py:28: UserWarning: The verbose parameter is deprecated. Please use get_last_lr() to access the learning rate.
  warnings.warn("The verbose parameter is deprecated. Please use get_last_lr() "
/Users/tonylizza/opt/anaconda3/envs/ML_new/lib/python3.12/site-packages/pytorch_lightning/core/optimizer.py:316: The lr scheduler dict contains the key(s) ['monitor'], but the keys will be ignored. You need to call `lr_scheduler.step()` manually in manual optimization.
/Users/tonylizza/opt/anaconda3/envs/ML_new/lib/python3.12/site-packages/pytorch_lightning/utilities/model_summary/mode

Epoch 0:   0%|          | 0/4 [00:00<?, ?it/s] 

/Users/tonylizza/Documents/Machine_Learning/Thesis_Coding/gflownet-spai/gflownet/dataset.py:39: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/miniforge3/conda-bld/libtorch_1719361045918/work/torch/csrc/utils/tensor_new.cpp:277.)
  ilu_indices = torch.tensor([ilu_matrix.row, ilu_matrix.col], dtype=torch.long)
/Users/tonylizza/Documents/Machine_Learning/Thesis_Coding/gflownet-spai/gflownet/dataset.py:39: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at /Users/runner/miniforge3/conda-bld/libtorch_1719361045918/work/torch/csrc/utils/tensor_new.cpp:277.)
  ilu_indices = torch.tensor([ilu_matrix.row, ilu_matrix.col], dtype=torch.long)
/Users/ton

[Finished Sample] CPU Memory Usage: 345.25 MB; VMS: 35926.68 MB
All Grad: True
Flows grad: None


/Users/tonylizza/opt/anaconda3/envs/ML_new/lib/python3.12/site-packages/pytorch_lightning/utilities/data.py:78: Trying to infer the `batch_size` from an ambiguous collection. The batch size we found is 18. To avoid any miscalculations, use `self.log(..., batch_size=batch_size)`.
/Users/tonylizza/Documents/Machine_Learning/Thesis_Coding/gflownet-spai/gflownet/gflownet.py:188: UserWarning: The .grad attribute of a Tensor that is not a leaf Tensor is being accessed. Its .grad attribute won't be populated during autograd.backward(). If you indeed want the .grad field to be populated for a non-leaf Tensor, use .retain_grad() on the non-leaf Tensor. If you access the non-leaf Tensor by mistake, make sure you access the leaf Tensor instead. See github.com/pytorch/pytorch/pull/30531 for more informations. (Triggered internally at /Users/runner/miniforge3/conda-bld/libtorch_1719361045918/work/build/aten/src/ATen/core/TensorBody.h:494.)
  print(f"Flows grad: {flows.grad}")


...

Type of b_vector: <class 'numpy.ndarray'>
GMRES converged successfully.
GMRES no preconditioner
GMRES did not converge. Exit code: tensor([30])
GMRES orig preconditioner
GMRES did not converge. Exit code: tensor([30])
GMRES sparse preconditioner
GMRES did not converge. Exit code: tensor([30])
GMRES sparse preconditioner
GMRES did not converge. Exit code: tensor([30])
GMRES sparse preconditioner
GMRES did not converge. Exit code: tensor([30])
GMRES sparse preconditioner
GMRES did not converge. Exit code: tensor([30])
GMRES sparse preconditioner
GMRES did not converge. Exit code: tensor([30])
GMRES sparse preconditioner
GMRES did not converge. Exit code: tensor([30])
GMRES sparse preconditioner
GMRES did not converge. Exit code: tensor([30])
GMRES sparse preconditioner
GMRES did not converge. Exit code: tensor([30])
GMRES sparse preconditioner


100%|██████████| 1/1 [22:31<00:00, 1351.29s/it]

GMRES did not converge. Exit code: tensor([30])
GMRES sparse preconditioner


100%|██████████| 1/1 [22:44<00:00, 1364.77s/it]


Validation results saved to validation_results_20241101160619.csv
Logged validation results after training epoch 11
Elapsed Training Time: 2666.687586069107
